[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/odd_elasticity/odd_elasticity.ipynb)

# Odd Elasticity

Odd elasticity is the elasticity of powered materials whose bonds push sideways when stretched. Such a material can deliver net work while returning to the same shape, and a lattice of such bonds carries waves even without inertia, every particle crawling through a viscous medium. The construction follows [Scheibner and collaborators, *Odd Elasticity*](https://arxiv.org/abs/1902.07760).

Positions and displacements are directions, antivectors, and a force is a vector, extrinsic, read by its pairing with a displacement: the work of a force over a displacement is `force & displacement`. A bond's response maps displacements to forces, `Force <- Displacement`: it reads the extension along the bond, but pushes partly across it. The adjugate of a map of displacements carries forces back through it with their work kept: from bonds to the shears that move them, and to the particles at their ends.

Three animations follow: one bond whose end follows a loop, a patch sheared around a cycle, and a wave crawling through a lattice.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_animation
from examples.mechanics.odd_elasticity import core, render

np.set_printoptions(precision=4, suppress=True)

In [ ]:
# The Euclidean algebra of the plane, x+y+.
ga = core.ga
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                           # Scalar+
# Positions and displacements are antivectors; a force is a vector, read by its pairing with a displacement.
Displacement = ga.gatype.antivector()                 # Antivector-
Force = ga.gatype.vector()                            # Vector+
Bivector = ga.gatype.bivector()                       # Bivector+
# A response maps displacements to forces.
Response = ga.gatype((Force, Displacement))
# The two amplitudes of a shear, geometric like displacements.
Shear = ga.gatype.antivector()                        # Antivector-

# The unit displacements along the two axes, which the unit forces `mv.x` and `mv.y` read as one.
x, y = stack((mv.x, mv.y)).dual()                     # [] Displacement-, Displacement-

## 1. Work from one bond

Stretching the bond produces a sideways force, orange, as well as a restoring one. Its end follows the pale loop and returns to its start, while the shaded region grows with the area it has swept around its rest position. The oriented actuation plane in which the push turns gives the material its handedness.

In [ ]:
stiffness = 1
activity = 3
rest_length = 1
parallel_radius = 0.25
transverse_radius = 0.18
n_steps = 72
duration_ms = 70
force_scale = 0.55
phases = np.linspace(0, 2 * np.pi, n_steps + 1)
direction = x                                                    # Displacement-
actuation_plane = mv.xy                                          # Bivector+

In [ ]:
def bond_response(direction: Displacement, actuation_plane: Bivector, stiffness: float, activity: float) -> Response:
    """The restoring and sideways force caused by a bond's endpoint displacement."""
    # The unit force along the bond, which reads a displacement's component along it.
    unit_force = direction.dual_inverse()                        # Force+
    push = -stiffness * unit_force + activity * (unit_force | actuation_plane)   # Force+
    return push * (unit_force & Displacement)                    # Force+ <- Displacement-

def cycle_states(response: Response, cycle: Displacement) -> Iterator[tuple[Displacement, Force, Scalar]]:
    """Displacement, force and work delivered along consecutive straight path segments."""
    work = mv.scalar([0])                                        # Scalar+
    yield cycle[0], response(cycle[0]), work
    for start, end in zip(cycle[:-1], cycle[1:]):
        # For a linear force, the midpoint gives the exact work on each straight segment.
        work = work + (response((start + end) / 2) & (end - start))   # Scalar+
        yield end, response(end), work


anchor = 0 * direction                                           # Displacement-
reference = rest_length * direction                              # Displacement-
response = bond_response(direction, actuation_plane, stiffness, activity)   # Force+ <- Displacement-
across = direction | actuation_plane                             # Displacement-
# The ellipse stretches the unit circle by its two radii, along the bond and across it.
ellipse = (parallel_radius * direction * (direction.dual_inverse() & Displacement)
           + transverse_radius * across * (across.dual_inverse() & Displacement))   # Displacement- <- Displacement-
cycle = ellipse((-actuation_plane * phases / 2).exp() >> direction)   # [steps + 1] Displacement-
bond_states = [(reference + displacement, force, work)
               for displacement, force, work in cycle_states(response, cycle)]
show_animation(render.bond_frames(anchor, reference, reference + cycle,
                                  [state[:2] for state in bond_states], force_scale),
               "odd_elasticity_bond", duration_ms)

The sideways push makes the work of a small displacement depend on the path, not only on where it ends. A passive bond's response is symmetric under the pairing and stores energy it gives back in full; the active push has an antisymmetric part, the odd part that gives odd elasticity its name, and that part delivers net work proportional to the oriented area the loop encloses. Running the loop backwards reverses the work.

In [ ]:
passive = bond_response(direction, actuation_plane, stiffness, 0)   # Force+ <- Displacement-
increments = cycle[1:] - cycle[:-1]                              # [steps] Displacement-
midpoints = (cycle[1:] + cycle[:-1]) / 2                         # [steps] Displacement-
work = (response(midpoints) & increments).sum(axis=0)            # Scalar+
passive_work = (passive(midpoints) & increments).sum(axis=0)     # Scalar+
enclosed_area = (cycle[:-1] ^ cycle[1:]).sum(axis=0) / 2         # Bivector+
area_work = -activity * actuation_plane.scalar_product(enclosed_area)   # Scalar+

In [ ]:
print("work around the loop:   ", work.to_array())
print("activity times its area:", area_work.to_array())
print("passive bond's work:    ", passive_work.to_array())

## 2. A shear cycle

A patch can stretch along one axis while narrowing along the other, or do the same along the diagonals. Cycling these two shears closes its deformation while delivering work. The adjugate of the map from shear amplitudes to bond displacements, `bond_motion.adjugate()`, returns the bond forces to the shear amplitudes with their work kept. The orange arrows show the forces the deformed patch exerts at its boundary; the work per lattice cell, which holds one bond of each of the three families, is again the activity times the area of the loop of shear amplitudes, three times over.

In [ ]:
spacing = 1
amplitude = 0.14
n_columns, n_rows = 7, 7
force_scale = 1.3
bond_angles = np.arange(3) * np.pi / 3
reference, edges, incidence = core.triangular_patch(n_columns, n_rows, spacing)

In [ ]:
# The two shear amplitudes stretch along the axes, or along the diagonals between them.
horizontal_shear = x * (mv.x & Displacement) - y * (mv.y & Displacement)   # Displacement- <- Displacement-
diagonal_shear = x * (mv.y & Displacement) + y * (mv.x & Displacement)   # Displacement- <- Displacement-
strain = (mv.x & Shear) * horizontal_shear + (mv.y & Shear) * diagonal_shear   # Displacement- <- (Shear-, Displacement-)
# One bond of each of the three families in a lattice cell.
bond_directions = (-actuation_plane * bond_angles / 2).exp() >> x   # [families] Displacement-
cell_response = bond_response(bond_directions, actuation_plane, stiffness, activity)   # [families] Force+ <- Displacement-
bond_motion = strain(Shear, spacing * bond_directions)           # [families] Displacement- <- Shear-
# A shear moves the bonds; their response supplies forces; the adjugate returns the loads on that shear.
shear_response = bond_motion.adjugate()(cell_response(bond_motion)).sum(axis=0)   # Force+ <- Shear-

# Summing the same bond forces over the finite patch reveals its boundary loads.
gradient = incidence * Displacement                              # (Displacement- <- Displacement-)[bonds <- sites]
edge_response = bond_response(gradient(reference).normalized(), actuation_plane, stiffness, activity)   # (Force+ <- Displacement-)[bonds]
patch_response = gradient.adjugate()(spdiag(edge_response)(gradient))   # (Force+ <- Displacement-)[sites <- sites]
patch_motion = strain(Shear, reference)                          # (Displacement- <- Shear-)[sites]
patch_forces = patch_response(patch_motion)                      # (Force+ <- Shear-)[sites]
shear_cycle = amplitude * ((-actuation_plane * phases / 2).exp() >> x)   # [steps + 1] Shear-
shear_states = [(reference + patch_motion(shear), patch_forces(shear), work)
                for shear, _, work in cycle_states(shear_response, shear_cycle)]
show_animation(render.patch_frames(reference, edges, [state[:2] for state in shear_states], force_scale),
               "odd_elasticity_shear", duration_ms)

## 3. Waves without inertia

A localized displacement is released into a triangular lattice. Every particle moves at the velocity its force sets against drag, nothing coasts, and since a velocity is a direction and a force a vector, drag carries one into the other through the metric, `Force.dual() / drags`. The sideways push couples stretching to motion across the bonds, and the lattice answers with damped oscillations. Blue bonds are compressed and orange ones extended, the gold particle traces its motion, and the faint mesh marks the rest positions, with displacements magnified four times.

In [ ]:
n_columns, n_rows = 21, 11
wave_stiffness = 0.2
wave_activity = 4
drag = 1
wave_amplitude = 0.15
width = 2.5
wave_number = 1.2
n_frames = 100
dt = 0.06
displacement_scale = 4
extension_scale = 0.04
tracked_index = (n_rows // 2) * n_columns + n_columns // 2
reference, edges, incidence = core.triangular_patch(n_columns, n_rows, spacing)

In [ ]:
def evolution(incidence: SparseExtensor, response: Response, drags: Scalar) -> SparseExtensor:
    """The particles' velocities from their displacements, overdamped: no inertia, every force met by drag."""
    gradient = incidence * Displacement                          # (Displacement- <- Displacement-)[bonds <- sites]
    # Drag turns each particle's force into its velocity, along the force's metric direction.
    mobility = spdiag(Force.dual() / drags)                      # (Velocity- <- Force+)[sites <- sites]
    # Relative endpoint motion goes to each bond; equal opposite forces return to its endpoints.
    return mobility(gradient.adjugate()(spdiag(response)(gradient)))   # (Velocity- <- Displacement-)[sites <- sites]


wave_directions = (incidence * reference).normalized()           # Displacement-[bonds]
wave_response = bond_response(wave_directions, actuation_plane, wave_stiffness, wave_activity)   # (Force+ <- Displacement-)[bonds]
drags = mv.scalar([[drag]] * n_columns * n_rows).field()         # Scalar+[sites]
velocities = evolution(incidence, wave_response, drags)          # (Velocity- <- Displacement-)[sites <- sites]
envelope = (-reference.scalar_norm_squared() / (2 * width**2)).exp()   # Scalar+[sites]
initial = wave_amplitude * x * envelope * (wave_number * (mv.x & reference)).cos()   # Displacement-[sites]


def wave_states(displacement: Displacement) -> Iterator[tuple[Displacement, Scalar]]:
    for _ in range(n_frames):
        # Each bond's extension, read by the unit force along it.
        extensions = wave_directions.dual_inverse() & (incidence * displacement)   # Scalar+[bonds]
        yield reference + displacement, extensions
        displacement = core.rk4_step(velocities, displacement, dt)   # Displacement-[sites]


show_animation(render.wave_frames(reference, edges, wave_states(initial), displacement_scale,
                                  extension_scale, tracked_index), "odd_elasticity_wave", duration_ms)

In [ ]:
# checks
# Around each loop the bonds deliver the activity times the enclosed area, oriented by the actuation
# plane; the passive bond delivers none.
shear_area = (shear_cycle[:-1] ^ shear_cycle[1:]).sum(axis=0) / 2   # Bivector+
shear_work = -3 * activity * actuation_plane.scalar_product(shear_area)   # Scalar+
np.testing.assert_allclose(bond_states[-1][2].to_array(), area_work.to_array(), atol=1e-12)
np.testing.assert_allclose(passive_work.to_array(), 0, atol=1e-12)
np.testing.assert_allclose(shear_states[-1][2].to_array(), shear_work.to_array(), atol=1e-12)

The same response has passed from a single bond to shear amplitudes and to a field of particle displacements. Each time, a map of displacements carries motion to the bonds and its adjugate carries their forces back. The bond response between them is not symmetric under the pairing, so the assembled material cannot be described by an elastic energy alone.